In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import sys
sys.path.append("../utils")
from plotting_utils import format_top_3, plot_metric_grouped_by, plot_bio_vs_batch_correction

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
base_path = ".."
mali_path = "../../MALI"
scratch_path = ".."

# save_name = f"ablation_labels_bad_kema/9092696" #dataset}".format(dataset = dataset_name)
# save_path = f"{scratch_path}/results/{save_name}"


load results

In [ ]:
exp = "ablation_labels/9110781"
results_path = f"../results/{exp}"

# load results
results_df = pd.read_csv(f"{results_path}/simulated_multimodal_results.csv")

# in the df, replace "_" with " " 
results_df['split'] = results_df['split'].str.replace('_', ' ')
results_df.columns = results_df.columns.str.replace('_', ' ')

# # convert relevant columns to numeric
num_cols = results_df.columns.difference(['model', 'dataset', 'split', 'seed'])
results_df[num_cols] = results_df[num_cols].apply(pd.to_numeric, errors='coerce')

# take the abs for silhouette domain because it should be low, doesnt matter the sign
results_df['Silhouette domain'] = results_df['Silhouette domain'].abs()

# remove RFMALI
# methods_to_remove= ["RFMALI"]
# results_df = results_df[~results_df["model"].isin(methods_to_remove)]
results_df

In [ ]:
results_df.value_counts(["split"], sort=True)

In [ ]:
results_df["split"].unique()

average over all datasets

In [ ]:
summary_df = results_df.groupby(['split','model', 'label mask fraction'])[num_cols].mean()
summary_df_std = results_df.groupby(['split','model', "label mask fraction"])[num_cols].std()
summary_df

In [ ]:
# clean summary_df 

def clean_summary_df(df):
    # remove "RFMALI" 
    df = df.loc[df.index.get_level_values('model') != 'RFMALI']

    cols_to_keep = ["Accuracy missing", "Alignment score", "FOSCTTM"]
    # rename Accuracy missing to Accuracy
    df = df[cols_to_keep]
    df = df.rename(columns={"Accuracy missing": "Accuracy"})
    return df

summary_df = clean_summary_df(summary_df)
summary_df_std = clean_summary_df(summary_df_std)

In [ ]:
summary_df_fmt = format_top_3(summary_df)

import warnings
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    print(summary_df_fmt.to_latex(
        escape=False,
        multirow=True,
        float_format="%.3f"
    )) 

# then copy the output latex table into a .tex file for inclusion in the paper

# line plots

In [ ]:
summary_df
cols_to_plot = ["Accuracy missing", "Alignment score", "FOSCTTM", "Silhouette domain"]
for col in cols_to_plot:
    plt.figure(figsize=(10,6))
    sns.lineplot(data=results_df, x='label mask fraction', y=col, hue='model')
    # sns.scatterplot(data=results_df, x='label mask fraction', y=col, hue='model')
    
    # if col == "Silhouette domain":
    #     plt.figure(figsize=(10,6))
    #     sns.lineplot(data=results_df[(results_df['model'] != "KEMArbf") & (results_df['model'] != "KEMAlin")], x='label mask fraction', y=col, hue='model')
    #     plt.show()

# memory and time

In [ ]:
# subset to 0.5 label mask fraction
subset_df = results_df[results_df['label mask fraction'] == 0.5]

for col in num_cols:
    plt.figure(figsize=(10,6))
    sns.barplot(data=subset_df, x='model', y=col, hue='split')
    
    
    if col == "Silhouette domain":
        plt.figure(figsize=(10,6))
        sns.barplot(data=subset_df, x='model', y=col, hue='split')
        plt.show()